# Device Management

Demo notebook for [PR #101](https://github.com/evo-design/bio-programming-tools/pull/101). Walks through automatic GPU allocation, LRU eviction, and multi-GPU routing.

In [1]:
from bio_programming_tools.tools.structure_prediction.esmfold import (
    run_esmfold,
    ESMFoldInput,
    ESMFoldConfig,
)
from bio_programming_tools.utils.tool_instance import ToolInstance
from bio_programming_tools.utils.device_manager import DeviceManager, OffloadStrategy
from bio_programming_tools.utils import display_gpu_memory_usage
from time import time, sleep

# Test sequences
TEST_SEQ = "MKTLLILAVVAAALA"
TEST_SEQ2 = "GAVLTVLLGGLLLA"

# Summary

Tools request devices through the config field `device`. Under the hood, the **DeviceManager** handles everything automatically:

- **Automatic GPU allocation** — just use `ToolInstance.persist()` and DeviceManager picks the right device
- **LRU eviction** — when GPUs are full, least recently used models are offloaded to CPU
- **Multi-GPU support** — named instances for parallel execution across GPUs

---
## 1. Requesting a Device

Every tool config has a `device` field. GPU tools default to `"cuda"`, CPU tools default to `"cpu"`. Device requests fall into two categories:

### General requests (DeviceManager chooses the GPU)

With a general request, DeviceManager finds available GPUs, handles allocation, and triggers LRU eviction if everything is busy. This is the recommended approach — you don't need to know which GPUs are free.

| Value | Meaning |
|---|---|
| `"cpu"` | Run on CPU |
| `"cuda"` | Run on one GPU — DeviceManager picks which one |
| `"cudax2"`, `"cudax3"`, ... | Run on N GPUs — DeviceManager picks which ones (for multi-gpu models) |

### Specific requests (you choose the GPU)
With a specific request, DeviceManager will place the model on exactly the device(s) you asked for. If that device is already occupied, it evicts whatever is there. Use this when you need precise control over which physical GPU a model lands on.

| Value | Meaning |
|---|---|
| `"cuda:0"` | Run on exactly GPU 0 |
| `"cuda:0,1"` or `"cuda:0,cuda:1"` | Run on exactly GPUs 0 and 1 (for multi-gpu models) |

### Example 1a: General vs Specific Allocation

With a **general** request (`device="cuda"`), DeviceManager picks an available GPU for you. With a **specific** request (`device="cuda:2"`), you choose exactly which GPU the model lands on.

In [2]:
print("Before:")
display_gpu_memory_usage()

# --- General request: DeviceManager picks the GPU ---
print("\n=== General request: device='cuda' ===")
with ToolInstance.persist():

    # NOTE: By default, a general "cuda" request is made for GPU devices.
    # You don't even have to pass in the config
    output = run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]))
    print("Model loaded:")
    display_gpu_memory_usage()

print("\n(cleaned up)")
display_gpu_memory_usage()

# --- Specific request: we pick the GPU ---
print("\n=== Specific request: device='cuda:2' ===")
with ToolInstance.persist():
    output = run_esmfold(
        ESMFoldInput(complexes=[TEST_SEQ]),
        ESMFoldConfig(device="cuda:2"),
    )
    print("Model loaded:")
    display_gpu_memory_usage()

print("\n(cleaned up)")
display_gpu_memory_usage()

Before:
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== General request: device='cuda' ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:35<00:00, 35.19s/structure]


Model loaded:
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

(cleaned up)
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Specific request: device='cuda:2' ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:17<00:00, 17.77s/structure]


Model loaded:
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:  15.0 /  85.5 GB  ██░░░░░░░░░░░░

(cleaned up)
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░


### Example 1b: Limiting Managed Devices

By default, DeviceManager auto-detects all `CUDA_VISIBLE_DEVICES` and allocates from the full pool. You can restrict which GPUs it uses with `configure(managed_devices=...)` or the `BIO_TOOLS_MANAGED_DEVICES` environment variable.

This is useful when you want to reserve some GPUs for other work, or only use a subset of a large machine.

In [3]:
# Default: DeviceManager sees all GPUs
dm = DeviceManager.get_instance()
print("Default pool:", dm.get_device_status()["available_devices"])

# Restrict to just two GPUs
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:1"])
print("Restricted pool:", dm.get_device_status()["available_devices"])

# General requests now only land on cuda:1
with ToolInstance.persist():
    output = run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]))
    print("\nModel loaded (general request with restricted pool):")
    display_gpu_memory_usage()

# Equivalent via environment variable:
# export BIO_TOOLS_MANAGED_DEVICES="cuda:1"

# Reset back to full pool
DeviceManager.reset_instance()

Default pool: ['cuda:0', 'cuda:1', 'cuda:2']
Restricted pool: ['cuda:1']


Folding structures (ESMFold): 100%|██████████| 1/1 [00:16<00:00, 16.07s/structure]



Model loaded (general request with restricted pool):
GPU 0:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 1:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░


---
## 2. Eviction Strategies

When all GPUs are full and a new model needs to load, DeviceManager evicts the **least recently used** model. This works across any number of GPUs — DeviceManager always picks the LRU model globally, regardless of which device it's on.

There are two strategies for what happens to the evicted model:

- **RESTART** (default) — the model's worker is shut down entirely. Frees all memory, but reloading later pays the full startup cost.
- **CPU** — the model is moved to CPU RAM. It stays loaded and can be moved back to GPU quickly, but uses system memory.

### Example 2a: RESTART (Default)

The evicted model is shut down entirely. GPU memory is fully freed, but the next call reloads from scratch.

In [4]:
# Limit to 1 device so eviction is guaranteed
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0"])

def show_status():
    """Show allocations + GPU memory in one block."""
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    if not allocs:
        print("  (no allocations)")
    display_gpu_memory_usage()

print("=== Load instance A ===")
with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
    show_status()

    print("\n=== Load instance B (RESTART evicts A — fully shut down) ===")
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
        show_status()

DeviceManager.reset_instance()

=== Load instance A ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:15<00:00, 15.43s/structure]


  A: cuda:0
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load instance B (RESTART evicts A — fully shut down) ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:16<00:00, 16.15s/structure]


  B: cuda:0
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░


### Example 2b: CPU Offload

With CPU strategy, the evicted model stays in RAM instead of being shut down. Moving it back to GPU is fast.

In [5]:
# Configure CPU offload strategy with 1 device
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0"], offload_strategy=OffloadStrategy.CPU)

def show_status():
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    display_gpu_memory_usage()

print("=== Load instance A ===")
with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
    show_status()

    print("\n=== Load instance B (evicts A to CPU) ===")
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
        show_status()

        print("\n=== Run A again (fast CPU → GPU reload, evicts B to CPU) ===")
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
        show_status()

DeviceManager.reset_instance()

=== Load instance A ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:14<00:00, 14.77s/structure]


  A: cuda:0
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load instance B (evicts A to CPU) ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:17<00:00, 17.59s/structure]


  A: cpu
  B: cuda:0
GPU 0:  15.9 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Run A again (fast CPU → GPU reload, evicts B to CPU) ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:06<00:00,  6.86s/structure]


  A: cuda:0
  B: cpu
GPU 0:  10.2 /  85.5 GB  █░░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░


### Example 2c: LRU Eviction Across Multiple GPUs

With 2 GPUs and 3 instances, the first two fill both GPUs. The third triggers LRU eviction of whichever was used least recently.

In [6]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0", "cuda:1"])

def show_status():
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    if not allocs:
        print("  (no allocations)")
    display_gpu_memory_usage()

print("=== Load A (gets cuda:0) ===")
with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
    show_status()

    print("\n=== Load B (gets cuda:1) ===")
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        sleep(0.1)
        run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
        show_status()

        print("\n=== Load C (evicts A — the LRU — from cuda:0) ===")
        with ToolInstance.persist_tool("esmfold", instance_name="C"):
            sleep(0.1)
            run_esmfold(ESMFoldInput(complexes=["MGQQPGKVLGDQRR"]), instance="C")
            show_status()

DeviceManager.reset_instance()

=== Load A (gets cuda:0) ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:15<00:00, 15.16s/structure]


  A: cuda:0
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load B (gets cuda:1) ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:12<00:00, 12.47s/structure]


  A: cuda:0
  B: cuda:1
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load C (evicts A — the LRU — from cuda:0) ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:13<00:00, 13.63s/structure]


  B: cuda:1
  C: cuda:0
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░


---
## 3. Multiple Models Per Device

By default, DeviceManager allows **one model per GPU** — loading a second model triggers eviction. If you have enough GPU memory to fit multiple models, enable `allow_multiple_per_device=True` to pack them together.

**Note:** DeviceManager doesn't currently track model sizes or estimate memory usage — it's on you to make sure the models you're packing actually fit. If you overcommit, you'll get an OOM error. Memory-aware packing is something we may add later, but it's non-trivial: other processes can leave GPU memory residue on devices, some tools are CLI-based so their memory footprint isn't directly observable, and memory utilization can scale with input size.

### Example 3a: Packing 4 Instances on 2 GPUs

With `allow_multiple_per_device=True`, instances round-robin across the available GPUs instead of evicting.

In [7]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0", "cuda:1"], allow_multiple_per_device=True)

def show_status():
    allocs = dm.get_device_status()["allocations"]
    for name, info in allocs.items():
        print(f"  {name}: {info['device_id']}")
    display_gpu_memory_usage()

with ToolInstance.persist_tool("esmfold", instance_name="A"):
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        with ToolInstance.persist_tool("esmfold", instance_name="C"):
            with ToolInstance.persist_tool("esmfold", instance_name="D"):

                print("=== Load A ===")
                run_esmfold(ESMFoldInput(complexes=[TEST_SEQ]), instance="A")
                show_status()

                print("\n=== Load B ===")
                run_esmfold(ESMFoldInput(complexes=[TEST_SEQ2]), instance="B")
                show_status()

                print("\n=== Load C ===")
                run_esmfold(ESMFoldInput(complexes=["MGQQPGKVLGDQRR"]), instance="C")
                show_status()

                print("\n=== Load D ===")
                run_esmfold(ESMFoldInput(complexes=["ASTVKFLGPVLDAA"]), instance="D")
                show_status()

                print("\n4 instances, 2 GPUs, no evictions!")

DeviceManager.reset_instance()

=== Load A ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:15<00:00, 15.20s/structure]


  A: cuda:0
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load B ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:12<00:00, 12.48s/structure]


  A: cuda:0
  B: cuda:1
GPU 0:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 1:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load C ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:13<00:00, 13.22s/structure]


  A: cuda:0
  B: cuda:1
  C: cuda:0
GPU 0:  30.1 /  85.5 GB  ████░░░░░░░░░░
GPU 1:  15.0 /  85.5 GB  ██░░░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

=== Load D ===


Folding structures (ESMFold): 100%|██████████| 1/1 [00:13<00:00, 13.50s/structure]


  A: cuda:0
  B: cuda:1
  C: cuda:0
  D: cuda:1
GPU 0:  30.1 /  85.5 GB  ████░░░░░░░░░░
GPU 1:  30.1 /  85.5 GB  ████░░░░░░░░░░
GPU 2:   0.0 /  85.5 GB  ░░░░░░░░░░░░░░

4 instances, 2 GPUs, no evictions!
